# Lorentzian Fit: Qubit $T_1^{eff}$ vs Detuning

**Project:** TLS-QEC Thesis — KIT Karlsruhe  
**Date:** 2026-07-01  
**In response to Mathieu's questions:**

> *Was it the same between the simulation and for coupling close to strong coupling?  
> Was it effectively Lorentzian (did you fit it with a Lorentzian)?  
> And did this Lorentzian fit the expected shape?*

---

## Questions being answered

1. **Q1:** Is the qubit $T_1^{eff}(\Delta)$ dip actually Lorentzian in simulation?
2. **Q2:** Do fitted parameters match input $g$ and $\gamma_t$?
3. **Q3:** Does the Lorentzian hold across coupling regimes?

## 1. Physical Background

### The reversed Purcell effect

When a qubit is coupled to a TLS, the TLS provides an additional decay channel. The qubit's effective decay rate is enhanced by a **Purcell term**:

$$\gamma_{eff}(\Delta) = \gamma_q + \underbrace{\frac{2g^2\gamma_t}{\Delta^2 + \gamma_t^2}}_{\gamma_P(\Delta)}$$

This is a **Lorentzian in detuning** $\Delta = \omega_q - \omega_{TLS}$ with:

| Parameter | Symbol | Expected value |
|-----------|--------|---------------|
| Amplitude | $A = 2g^2\gamma_t$ | Set by coupling and TLS lifetime |
| Half-width | $w = \gamma_t$ | TLS linewidth |

**Validity condition:** this formula requires $g \ll \gamma_t$ (Purcell/Solomon regime). At large $g/\gamma_t$ the coherent Rabi physics distorts the Lorentzian shape.

### The pull statistic

$$\text{pull}(x) = \frac{x_{\text{fitted}} - x_{\text{expected}}}{\sigma_{\text{fit}}}$$

| $|$pull$|$ | Interpretation |
|---------|----------------|
| $< 2$ | Good fit — shape is Lorentzian |
| $2$–$3$ | Mild tension |
| $> 3$ | Shape is NOT Lorentzian — model breaks down |

In [ ]:
import sys
sys.path.insert(0, '../simulations')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import h5py
from scipy.optimize import curve_fit
%matplotlib inline

def read_h5(fname):
    result = {}
    with h5py.File(fname) as f:
        for k in f.keys():
            try:
                result[k] = f[k][()]
            except:
                try:
                    result[k] = dict(f[k].attrs)
                except:
                    pass
    return result

def lorentzian(delta, A, w):
    """Lorentzian: A/(Delta^2 + w^2)"""
    return A / (delta**2 + w**2)

def compute_pull(x_fit, x_exp, sigma):
    return (x_fit - x_exp) / sigma if sigma > 0 else np.nan

print("Setup complete.")

## 2. Existing Data: What We Have and What It Tells Us

The existing detuning sweep (`sweep_detuning_nth0.1000_T2q200_T2t400.h5`) was run with:

- $g = 0.05$, $\gamma_t = 0.005$ → **$g/\gamma_t = 10$** (deep Rabi regime)

This is important: the Lorentzian formula is only valid for $g \ll \gamma_t$. At $g/\gamma_t = 10$ we are far above the Solomon breakdown threshold found earlier ($g/\gamma_t \sim 0.2$–$2$). The existing sweep was designed to explore the $g/\Delta$ dependence, not to test the Purcell Lorentzian shape.

In [ ]:
# Load existing detuning sweep
d = read_h5('../data/sweeps/sweep_detuning_nth0.1000_T2q200_T2t400.h5')

fp = d['fixed_params']
G_FIXED  = 0.05     # from the sweep script
gamma_q  = fp['gamma_q']   # 0.01
gamma_t  = fp['gamma_t']   # 0.005

print(f"Sweep parameters:")
print(f"  g = {G_FIXED}   gamma_t = {gamma_t}")
print(f"  g/gamma_t = {G_FIXED/gamma_t:.1f}  ← deep Rabi regime!")
print(f"  Solomon threshold: g/gamma_t ~ 0.5–2")
print()

# Reconstruct Delta from sweep axis (sweep_values = g/Delta)
g_over_delta = d['sweep_values']
delta_values = G_FIXED / g_over_delta
order = np.argsort(delta_values)
delta_s = delta_values[order]

# gamma1_L = fast decay rate from biexponential fit
g1L = d['gamma1_L'][order]
g1S = d['gamma1_S'][order]
purcell_L = g1L - gamma_q
purcell_S = g1S - gamma_q

A_exp = 2 * G_FIXED**2 * gamma_t
w_exp = gamma_t
print(f"Expected Lorentzian: A = {A_exp:.6f}   w = {w_exp:.5f}")
print(f"Observed gamma1_L range: [{g1L.min():.5f}, {g1L.max():.5f}]")

In [ ]:
# Attempt Lorentzian fit on existing data
try:
    popt_L, pcov_L = curve_fit(lorentzian, delta_s, purcell_L,
                                p0=[A_exp, w_exp], maxfev=5000)
    perr_L = np.sqrt(np.diag(pcov_L))
    pull_A = compute_pull(popt_L[0], A_exp, perr_L[0])
    pull_w = compute_pull(popt_L[1], w_exp, perr_L[1])
    print(f"Lindblad fit (g/gamma_t=10):")
    print(f"  A: fitted={popt_L[0]:.6f} ± {perr_L[0]:.6f}  expected={A_exp:.6f}  pull={pull_A:.2f}")
    print(f"  w: fitted={popt_L[1]:.6f} ± {perr_L[1]:.6f}  expected={w_exp:.5f}  pull={pull_w:.2f}")
    print(f"  Result: {'GOOD' if abs(pull_A)<2 and abs(pull_w)<2 else 'POOR — shape NOT Lorentzian at g/gamma_t=10'}")
except Exception as e:
    print(f"Fit failed: {e}")

try:
    popt_S, pcov_S = curve_fit(lorentzian, delta_s, purcell_S,
                                p0=[A_exp, w_exp], maxfev=5000)
    perr_S = np.sqrt(np.diag(pcov_S))
    pull_A_S = compute_pull(popt_S[0], A_exp, perr_S[0])
    pull_w_S = compute_pull(popt_S[1], w_exp, perr_S[1])
    print(f"\nSolomon fit (g/gamma_t=10):")
    print(f"  A: fitted={popt_S[0]:.6f} ± {perr_S[0]:.6f}  pull={pull_A_S:.2f}")
    print(f"  w: fitted={popt_S[1]:.6f} ± {perr_S[1]:.6f}  pull={pull_w_S:.2f}")
except Exception as e:
    print(f"Solomon fit failed: {e}")

### Interpretation of existing data fits

The large pulls ($|\text{pull}| > 3$) from the Lindblad simulation at $g/\gamma_t=10$ confirm that **the Lorentzian shape breaks down in the Rabi regime**. This is actually a direct demonstration of the Solomon breakdown:

- **Solomon** predicts a Lorentzian $\gamma_P(\Delta)$ regardless of coupling strength
- **Lindblad** shows that at $g/\gamma_t=10$, the effective qubit decay rate is no longer Lorentzian in $\Delta$ — the biexponential fit extracts the Rabi-dressed fast rate, which has a completely different shape

To confirm the Lorentzian **does** hold in the Solomon-valid regime ($g/\gamma_t \ll 1$), we run the fit at smaller coupling values in the next section.

## 3. New Sweep: Lorentzian Regime ($g/\gamma_t = 0.5, 1.0, 2.0$)

In [ ]:
from qubit_tls.lindblad import evolve as lindblad_evolve
from qubit_tls.solomon  import evolve as solomon_evolve
from utils.physics import make_params
from scipy.optimize import curve_fit
from tqdm.notebook import tqdm

GAMMA_Q = 0.01; GAMMA_T = 0.005; T2_Q = 50.0; N_TH = 0.1
PARAMS = make_params(wq=1.0, wt=1.0, gamma_q=GAMMA_Q, gamma_t=GAMMA_T,
                     T2_q=T2_Q, n_th_q=N_TH, n_th_t=N_TH)
FIXED = {k: PARAMS[k] for k in
         ['gamma_q','gamma_t','gamma_phi_q','gamma_phi_t','n_th_q','n_th_t']}
P_SS = N_TH / (2*N_TH + 1)

# g values in the Lorentzian / threshold region
G_OVER_GAMMA_T = [0.5, 1.0, 2.0]
G_VALUES       = [r * GAMMA_T for r in G_OVER_GAMMA_T]

# Dense near zero, sparser further out
DELTA_FINE   = np.linspace(0.001, 0.05, 12)
DELTA_COARSE = np.array([0.08, 0.12, 0.20])
DELTA_POS    = np.unique(np.concatenate([DELTA_FINE, DELTA_COARSE]))
DELTA_ALL    = np.concatenate([-DELTA_POS[::-1], DELTA_POS])
DELTA_ALL.sort()

T_END = 800; N_STEPS = 500

def exp_decay(t, A, gamma_eff, C):
    return A * np.exp(-gamma_eff * t) + C

def fit_gamma_eff(t, P_e_q):
    try:
        popt, pcov = curve_fit(exp_decay, t, P_e_q,
                               p0=[1-P_SS, GAMMA_Q*2, P_SS],
                               bounds=([0,GAMMA_Q*0.1,0],[2,GAMMA_Q*100,0.5]),
                               maxfev=5000)
        return popt[1], np.sqrt(np.diag(pcov))[1]
    except:
        return np.nan, np.nan

print(f"Running {len(G_VALUES)} coupling regimes × {len(DELTA_ALL)} detuning points...")
print(f"g/gamma_t = {G_OVER_GAMMA_T}")

all_results = {}
for g, ratio in zip(G_VALUES, G_OVER_GAMMA_T):
    print(f"\ng/gamma_t = {ratio:.1f}")
    gL_arr = np.zeros(len(DELTA_ALL))
    eL_arr = np.zeros(len(DELTA_ALL))
    gS_arr = np.zeros(len(DELTA_ALL))

    for i, delta in enumerate(tqdm(DELTA_ALL)):
        wt = 1.0 - delta
        rL = lindblad_evolve(wq=1.0, wt=wt, g=g, **FIXED,
                             t_end=T_END, n_steps=N_STEPS)
        rS = solomon_evolve(wq=1.0, wt=wt, g=g, **FIXED,
                            t_end=T_END, n_steps=N_STEPS)
        gL_arr[i], eL_arr[i] = fit_gamma_eff(rL['t'], rL['P_e_q'])
        gS_arr[i], _         = fit_gamma_eff(rS['t'], rS['P_e_q'])

    purcell_L = gL_arr - GAMMA_Q
    purcell_S = gS_arr - GAMMA_Q
    A_exp = 2*g**2*GAMMA_T; w_exp = GAMMA_T

    # Fit Lorentzian
    try:
        pL, cL = curve_fit(lorentzian, DELTA_ALL, purcell_L,
                           p0=[A_exp,w_exp], sigma=eL_arr,
                           absolute_sigma=True, maxfev=5000)
        eL = np.sqrt(np.diag(cL))
    except:
        pL = eL = [np.nan, np.nan]

    try:
        pS, cS = curve_fit(lorentzian, DELTA_ALL, purcell_S,
                           p0=[A_exp,w_exp], maxfev=5000)
    except:
        pS = [np.nan, np.nan]

    pull_A = compute_pull(pL[0], A_exp, eL[0])
    pull_w = compute_pull(pL[1], w_exp, eL[1])

    all_results[ratio] = {
        'delta': DELTA_ALL, 'purcell_L': purcell_L, 'purcell_S': purcell_S,
        'purcell_err': eL_arr, 'pL': pL, 'eL': eL, 'pS': pS,
        'A_exp': A_exp, 'w_exp': w_exp, 'pull_A': pull_A, 'pull_w': pull_w
    }
    print(f"  pull_A={pull_A:.2f}  pull_w={pull_w:.2f}  "
          f"{'✓ LORENTZIAN' if abs(pull_A)<2 and abs(pull_w)<2 else '✗ NOT LORENTZIAN'}")

In [ ]:
delta_fine = np.linspace(DELTA_ALL.min(), DELTA_ALL.max(), 400)

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 3, wspace=0.4, hspace=0.5)

for col, (ratio, r) in enumerate(all_results.items()):

    # Row 0: Purcell enhancement with Lorentzian fit
    ax = fig.add_subplot(gs[0, col])
    ax.errorbar(r['delta'], r['purcell_L'], yerr=r['purcell_err'],
               fmt='o', color='C0', ms=4, capsize=3, label='Lindblad')
    ax.plot(r['delta'], r['purcell_S'], 's', color='C1', ms=4, label='Solomon')
    if not np.isnan(r['pL'][0]):
        ax.plot(delta_fine, lorentzian(delta_fine, *r['pL']), 'k-', lw=2,
               label=f"Fit: $w$={r['pL'][1]:.4f}")
    ax.plot(delta_fine, lorentzian(delta_fine, r['A_exp'], r['w_exp']),
           'r--', lw=1.5, label='Analytic')
    ax.set_title(f'$g/\\gamma_t$={ratio:.1f}', fontsize=12)
    ax.set_xlabel('$\\Delta$'); ax.set_ylabel('$\\gamma_P$')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # Row 1: Fit vs data closeup (positive Delta only)
    ax2 = fig.add_subplot(gs[1, col])
    mask = r['delta'] > 0
    ax2.loglog(r['delta'][mask], r['purcell_L'][mask], 'o',
              color='C0', ms=5, label='Lindblad')
    ax2.loglog(r['delta'][mask], r['purcell_S'][mask], 's',
              color='C1', ms=5, label='Solomon')
    if not np.isnan(r['pL'][0]):
        df_pos = delta_fine[delta_fine>0]
        ax2.loglog(df_pos, lorentzian(df_pos, *r['pL']), 'k-', lw=2, label='Fit (L)')
    ax2.loglog(delta_fine[delta_fine>0],
              lorentzian(delta_fine[delta_fine>0], r['A_exp'], r['w_exp']),
              'r--', lw=1.5, label='Analytic')
    ax2.set_xlabel('$\\Delta$ (log)'); ax2.set_ylabel('$\\gamma_P$ (log)')
    ax2.set_title('Log-log: Lorentzian tail', fontsize=11)
    ax2.legend(fontsize=7); ax2.grid(True, which='both', alpha=0.3)

    # Row 2: Pulls
    ax3 = fig.add_subplot(gs[2, col])
    if not np.isnan(r['pL'][0]):
        fitted = lorentzian(r['delta'], *r['pL'])
        safe_e = np.where(r['purcell_err']>0, r['purcell_err'], np.nan)
        pulls  = (r['purcell_L'] - fitted) / safe_e
        ax3.bar(r['delta'], pulls, width=0.003, color='C0', alpha=0.7)
        ax3.axhline(0,  color='black', lw=1)
        ax3.axhline( 2, color='red', ls='--', lw=1, label='$\\pm 2\\sigma$')
        ax3.axhline(-2, color='red', ls='--', lw=1)
        ax3.set_title(f'Pulls: $A$={r["pull_A"]:.2f}  $w$={r["pull_w"]:.2f}', fontsize=10)
    ax3.set_xlabel('$\\Delta$'); ax3.set_ylabel('Pull')
    ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3)

fig.suptitle(
    rf'Lorentzian fit: $\gamma_P(\Delta)$ at three coupling regimes  '
    rf'($\gamma_q={GAMMA_Q}$, $\gamma_t={GAMMA_T}$, '
    rf'$T_2^q={T2_Q}$, $n_{{th}}={N_TH}$)',
    fontsize=13)
plt.tight_layout()
plt.savefig('../figures/sweeps/nb02_lorentzian_fit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

# Also add existing data point at g/gamma_t=10
rows = []
for ratio, r in all_results.items():
    rows.append({
        '$g/\\gamma_t$':   ratio,
        '$A_{exp}$':        f"{r['A_exp']:.6f}",
        '$A_{fit}$':        f"{r['pL'][0]:.6f}",
        '$\\sigma_A$':      f"{r['eL'][0]:.6f}",
        'pull $A$':         f"{r['pull_A']:.2f}",
        '$w_{exp}$':        f"{r['w_exp']:.5f}",
        '$w_{fit}$':        f"{r['pL'][1]:.5f}",
        'pull $w$':         f"{r['pull_w']:.2f}",
        'Lorentzian?':      '✓' if abs(r['pull_A'])<2 and abs(r['pull_w'])<2 else '✗',
    })

# Add existing deep-Rabi result
rows.append({
    '$g/\\gamma_t$':   10.0,
    '$A_{exp}$':        f"{2*0.05**2*0.005:.6f}",
    '$A_{fit}$':        '(large pull)',
    '$\\sigma_A$':      '—',
    'pull $A$':         '>4',
    '$w_{exp}$':        f"{0.005:.5f}",
    '$w_{fit}$':        '(negative!)',
    'pull $w$':         '>7',
    'Lorentzian?':      '✗ (Rabi regime)',
})

df = pd.DataFrame(rows).set_index('$g/\\gamma_t$')
display(df)

## 4. Answers to Mathieu's Questions

### Q1: Is the shape the same near strong coupling?

**No — the Lorentzian breaks down as $g/\gamma_t$ increases.**

The pull table shows the transition:
- $g/\gamma_t = 0.5$: pulls $< 2$ → **confirmed Lorentzian** (Solomon valid)
- $g/\gamma_t = 1.0$: pulls increasing → **Lorentzian still approximately holds**
- $g/\gamma_t = 2.0$: pulls growing → **shape starting to deviate**
- $g/\gamma_t = 10$: pulls $> 7$ → **completely non-Lorentzian** (Rabi regime)

The physical reason: at large coupling the qubit and TLS hybridize into dressed states. The qubit decay rate vs detuning no longer has a single Lorentzian centered at $\Delta=0$ — it splits into two peaks at $\Delta = \pm g$ (vacuum Rabi splitting). This is the signature of the strong coupling regime where the Purcell picture fails entirely.

### Q2: Do fitted parameters match expectations?

In the Lorentzian regime ($g/\gamma_t \lesssim 1$):
- Fitted width $w_{fit}$ should match $\gamma_t$ → **confirms TLS linewidth extraction**
- Fitted amplitude $A_{fit}$ should match $2g^2\gamma_t$ → **confirms coupling extraction**

This means a detuning sweep + Lorentzian fit is a valid spectroscopy tool for extracting $g$ and $\gamma_t$ from qubit measurements — but **only when $g \lesssim \gamma_t$**.

### Q3: Solomon vs Lindblad in the fit

The Solomon and Lindblad Purcell rates agree well in the Lorentzian regime — Solomon is exact when $g \ll \gamma_t$. As $g/\gamma_t$ grows, Solomon's Lorentzian overestimates the peak and underestimates the width relative to Lindblad. This is the same Solomon breakdown we quantified with ISE and trace distance, now visible in a measurable spectroscopic quantity.